In [1]:
!pip install selenium webdriver-manager pandas

In [2]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# --- Configurações do Pipeline ---

# 1. Diretório de Download (Stage Area)
DIRETORIO_DOWNLOAD = os.path.abspath(os.getcwd())
NOME_ARQUIVO_CSV = "status_invest_acoes.csv"
CAMINHO_ARQUIVO_COMPLETO = os.path.join(DIRETORIO_DOWNLOAD, NOME_ARQUIVO_CSV)

print(f"Diretório de Stage (download): {DIRETORIO_DOWNLOAD}")
print(f"Arquivo de destino: {CAMINHO_ARQUIVO_COMPLETO}")

# 2. Garantir que a área de stage está limpa
if os.path.exists(CAMINHO_ARQUIVO_COMPLETO):
    print("Arquivo antigo encontrado. Excluindo...")
    os.remove(CAMINHO_ARQUIVO_COMPLETO)

# 3. Configurar o Chrome + Download Automático
chrome_options = Options()
#chrome_options.add_argument("--headless") # Rodar em background
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")

# 4. Configurar o local de download
prefs = {
    "download.default_directory": DIRETORIO_DOWNLOAD,
    "download.prompt_for_download": False
}
chrome_options.add_experimental_option("prefs", prefs)

Diretório de Stage (download): C:\Users\vitor\projetos\invest-flow\mvp
Arquivo de destino: C:\Users\vitor\projetos\invest-flow\mvp\status_invest_acoes.csv
Arquivo antigo encontrado. Excluindo...


In [3]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
import glob

# --- Início da Extração ---
print("Iniciando driver... (Modo Visual para Debug)")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), 
                            options=chrome_options)

# Vamos manter os 20 segundos
TIMEOUT_MAXIMO = 10
wait = WebDriverWait(driver, TIMEOUT_MAXIMO)
wait_curto = WebDriverWait(driver, 5)

URL = 'https://statusinvest.com.br/acoes/busca-avancada'

try:
    print(f'Fazendo requisição para: {URL}')
    driver.get(URL)

    # --- ETAPA 1: Lidar com o Banner de Rodapé (Bloqueador 2) ---
    try:
        print("Aguardando Bloqueador 2 (Banner Rodapé)...")
        # Seletor baseado no HTML: <div id="footer-fixed"><a href="#">×</a>...</div>
        seletor_anuncio_footer = (By.XPATH, "//div[@id='footer-fixed']/a")
        
        wait_curto.until(EC.element_to_be_clickable(seletor_anuncio_footer)).click()
        print("Bloqueador 2 (Banner Rodapé) fechado.")
        time.sleep(0.5)
    except TimeoutException:
        print("Bloqueador 2 (Banner Rodapé) não encontrado. Seguindo...")

    # --- ETAPA 2: SCROLL e CLIQUE no "Buscar" ---
    print('Aguardando botão "Buscar"...')
    seletor_buscar = '//button[contains(@class, "find")]' 

    try:
        print("Procurando o botão 'Buscar' no DOM...")
        button_buscar = wait.until(
            EC.presence_of_element_located((By.XPATH, seletor_buscar))
        )
        
        print("Scrolleando o botão 'Buscar' para a tela...")
        driver.execute_script("arguments[0].scrollIntoView(true);", button_buscar)
        time.sleep(0.5) 

        print("Clicando em 'Buscar'...")
        wait.until(EC.element_to_be_clickable((By.XPATH, seletor_buscar))).click()
        
    except Exception as e:
        print(f"\n--- ERRO CRÍTICO (BUSCAR) ---: {type(e).__name__}")
        driver.save_screenshot("debug_erro_buscar.png")
        driver.quit()
        raise 

    # --- ETAPA 3 (NOVA ORDEM): Lidar com o Modal (Bloqueador 1) ---
    try:
        print("Aguardando Bloqueador 1 (Modal 'wisepops') pós-clique...")
        seletor_anuncio_modal = (By.XPATH, '//button[contains(@class, "wisepops-close") and contains(@class, "btn-close")]')
        
        wait_curto.until(EC.element_to_be_clickable(seletor_anuncio_modal)).click()
        print("Bloqueador 1 (Modal) fechado.")
        time.sleep(1)
        
    except TimeoutException:
        print("Nenhum 'wisepops' modal apareceu pós-clique. Ótimo!")


    # --- ETAPA 4: ESPERAR A TABELA DE RESULTADOS CARREGAR ---
    try:
        print("Aguardando a tabela de resultados carregar...")
        seletor_resultados = (By.ID, "list-result") #
        wait.until(EC.visibility_of_element_located(seletor_resultados))
        print("Tabela de resultados visível.")
    
    except TimeoutException:
        print("\n--- ERRO CRÍTICO (TABELA) ---")
        print("O script clicou em 'Buscar', mas a tabela de resultados não apareceu.")
        driver.save_screenshot("debug_erro_tabela.png")
        driver.quit()
        raise

    # --- ETAPA 5: Clicar em "Download" ---
    print('Aguardando botão "Download"...')
    seletor_download = '//a[contains(@class, "btn-download")]'
    
    try:
        button_download = wait.until(
            EC.element_to_be_clickable((By.XPATH, seletor_download))
        )
        print("Clicando em 'Download'...")
        button_download.click()

    except TimeoutException:
        print("\n--- ERRO CRÍTICO (DOWNLOAD) ---")
        driver.quit()
        raise

    # --- ETAPA 6: ESPERA DE DOWNLOAD ROBUSTA ---
    print(f"Vigiando pasta de download: {DIRETORIO_DOWNLOAD}")
    print("Aguardando o download ser concluído (timeout de 30s)...")
    
    tempo_limite = 15
    tempo_inicial = time.time()
    arquivo_encontrado = False
    
    while time.time() - tempo_inicial < tempo_limite:
        arquivos_em_progresso = glob.glob(os.path.join(DIRETORIO_DOWNLOAD, "*.crdownload"))
        if arquivos_em_progresso:
            time.sleep(1) 
            continue
            
        arquivos_csv = glob.glob(os.path.join(DIRETORIO_DOWNLOAD, "*.csv"))
        
        if arquivos_csv:
            caminho_arquivo_real = arquivos_csv[0]
            print(f"Download concluído. Arquivo encontrado: {caminho_arquivo_real}")
            
            try:
                os.rename(caminho_arquivo_real, CAMINHO_ARQUIVO_COMPLETO)
                print(f"Arquivo renomeado para: {NOME_ARQUIVO_CSV}")
                arquivo_encontrado = True
                break 
            except PermissionError:
                print("Arquivo encontrado, mas bloqueado (PermissionError). Tentando em 2s...")
                time.sleep(2)
                os.rename(caminho_arquivo_real, CAMINHO_ARQUIVO_COMPLETO)
                print(f"Arquivo renomeado para: {NOME_ARQUIVO_CSV}")
                arquivo_encontrado = True
                break 
        
        time.sleep(1)

    if not arquivo_encontrado:
        print("Timeout! O download não foi concluído em 30 segundos.")

finally:
    # 7. Sempre feche o driver
    print("Fechando o driver do Chrome.")
    driver.quit()

# --- Fim da Extração ---

# 8. Verificação final
if not os.path.exists(CAMINHO_ARQUIVO_COMPLETO):
    print("\n--- ERRO CRÍTICO ---")
    print(f"O script terminou, mas o arquivo CSV ({NOME_ARQUIVO_CSV}) não foi encontrado na pasta.")
else:
    print("\n--- SUCESSO ---")
    print(f"Arquivo CSV bruto pronto para Célula 4 em: {CAMINHO_ARQUIVO_COMPLETO}")

Iniciando driver... (Modo Visual para Debug)
Fazendo requisição para: https://statusinvest.com.br/acoes/busca-avancada
Aguardando Bloqueador 2 (Banner Rodapé)...
Bloqueador 2 (Banner Rodapé) fechado.
Aguardando botão "Buscar"...
Procurando o botão 'Buscar' no DOM...
Scrolleando o botão 'Buscar' para a tela...
Clicando em 'Buscar'...
Aguardando Bloqueador 1 (Modal 'wisepops') pós-clique...
Nenhum 'wisepops' modal apareceu pós-clique. Ótimo!
Aguardando a tabela de resultados carregar...
Tabela de resultados visível.
Aguardando botão "Download"...
Clicando em 'Download'...
Vigiando pasta de download: C:\Users\vitor\projetos\invest-flow\mvp
Aguardando o download ser concluído (timeout de 30s)...
Download concluído. Arquivo encontrado: C:\Users\vitor\projetos\invest-flow\mvp\statusinvest-busca-avancada.csv
Arquivo renomeado para: status_invest_acoes.csv
Fechando o driver do Chrome.

--- SUCESSO ---
Arquivo CSV bruto pronto para Célula 4 em: C:\Users\vitor\projetos\invest-flow\mvp\status_inv

In [4]:
import numpy as np
# --- Célula 4 ---

try:
    print(f"Lendo dados de stage: {CAMINHO_ARQUIVO_COMPLETO}")
    
    df_raw = pd.read_csv(
        CAMINHO_ARQUIVO_COMPLETO,
        sep=';',
        decimal=',',
        thousands='.'
    )
    print(f"Dados brutos lidos. {df_raw.shape[0]} linhas encontradas.")

    # --- INÍCIO DA TRANSFORMAÇÃO (T) ---
    df_clean = df_raw.copy()
    
    df_clean.columns = df_clean.columns.str.strip()
    print("Nomes de colunas limpos.")

    # 1. Definir as colunas-chave de métricas
    METRICAS_CHAVE = ['P/L', 'ROE', 'LIQ. CORRENTE', 'PRECO', 'DY', 'LIQUIDEZ MEDIA DIARIA', 'P/VP']

    # 2. Loop de transformação e limpeza
    for col in METRICAS_CHAVE:
        if col in df_clean.columns:
            # Primeiro, . 'errors=coerce' transforma 'N/A' ou vazios em NaN
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce') # Converte para numérico
            df_clean[col] = df_clean[col].fillna(0) # Substitui TODOS os NaNs
        else:
            print(f"Aviso: Coluna '{col}' não foi encontrada no CSV.")

    print("Tipagem de dados e tratamento de NaN concluídos.")

    # 3. Filtro (Screener)
    print("Aplicando filtros...")
    
    filtro_pl = (df_clean['P/L'] > 3) & (df_clean['P/L'] < 10)
    filtro_dy = (df_clean['DY'] > 6) & (df_clean['DY'] < 15)
    filtro_pvp = (df_clean['P/VP'] > 0.5) & (df_clean['P/VP'] < 2)
    filtro_lmd = (df_clean['LIQUIDEZ MEDIA DIARIA'] > 1000000)
    
    # 4. Carga (Load)
    df_filtrado = df_clean[filtro_pl & filtro_dy & filtro_pvp & filtro_lmd].copy()

    # 5. Cálculos da "Bola de Neve"
    print("Calculando DPA, Magic Number e Dividendo Gerado...")

    # 5a. Calcular DPA (Dividendo Por Ação em R$)
    # DPA = Preço * (DY / 100)
    df_filtrado['DPA (R$)'] = df_filtrado['PRECO'] * (df_filtrado['DY'] / 100)
    
    # 5b. Calcular Magic Number (N)
    # N = 100 / DY (arredondado para cima)
    df_filtrado['MAGIC_NUMBER'] = np.where(
        df_filtrado['DY'] > 0, 
        np.ceil(100 / df_filtrado['DY']), # np.ceil arredonda para cima
        np.inf
    )
    
    # 5c. Calcular Dividendo Gerado (Prova Real)
    # Dividendo Gerado = N * DPA
    df_filtrado['DIVIDENDO_GERADO (R$)'] = df_filtrado['MAGIC_NUMBER'] * df_filtrado['DPA (R$)']

    # 6. Ordenando a tabela
    df_filtrado = df_filtrado.sort_values(by='MAGIC_NUMBER')
    
    print("\n--- RESULTADO DO FILTRO ---")
    colunas_para_exibir = [
        'TICKER', 
        'PRECO', 
        'DY', 
        'P/L', 
        'P/VP',
        'MAGIC_NUMBER',
        'DIVIDENDO_GERADO (R$)'
    ]
    display(df_filtrado[colunas_para_exibir].head())


except FileNotFoundError:
    print(f"Erro: O arquivo CSV '{CAMINHO_ARQUIVO_COMPLETO}' não foi encontrado.")
except KeyError as e:
    print(f"Erro de Chave (KeyError): A coluna {e} não foi encontrada.")
    print("O schema do CSV do Status Invest mudou. Verifique 'df_raw.columns'.")
    display(df_raw.head())
except Exception as e:
    print(f"Erro inesperado ao processar o arquivo CSV: {e}")

Lendo dados de stage: C:\Users\vitor\projetos\invest-flow\mvp\status_invest_acoes.csv
Dados brutos lidos. 611 linhas encontradas.
Nomes de colunas limpos.
Tipagem de dados e tratamento de NaN concluídos.
Aplicando filtros...
Calculando DPA, Magic Number e Dividendo Gerado...

--- RESULTADO DO FILTRO ---


,TICKER,PRECO,DY,P/L,P/VP,MAGIC_NUMBER,DIVIDENDO_GERADO (R$)
155,CMIG4,10.72,13.65,4.72,1.08,8.0,11.706240
86,BRAP4,17.67,11.48,6.42,0.79,9.0,18.256644
295,GRND3,5.19,11.23,5.77,1.16,9.0,5.245533
375,MELK3,3.81,11.55,6.12,0.73,9.0,3.960495
595,VLID3,21.15,10.71,6.16,0.99,10.0,22.651650


In [27]:
# 5. Limpeza
if os.path.exists(CAMINHO_ARQUIVO_COMPLETO):
    print(f"Limpando área de stage... Excluindo {CAMINHO_ARQUIVO_COMPLETO}")
    os.remove(CAMINHO_ARQUIVO_COMPLETO)
    print("Arquivo bruto excluído com sucesso.")
else:
    print("Nenhum arquivo bruto para excluir.")

print("\nPipeline de protótipo finalizado.")

Limpando área de stage... Excluindo C:\Users\vitor\projetos\invest-flow\status_invest_acoes.csv
Arquivo bruto excluído com sucesso.

Pipeline de protótipo finalizado.
